<a href="https://colab.research.google.com/github/sr763/DICOM_Metadata_extraction_software/blob/main/Metadata_extraction_CT_MRT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
     # ======================================================================
# Code Title: CT_Metadata_extraction with Python
# Description: This script extracts metadata from DICOM format files. The source data is MRT and CT extracted from 1 patient, 1 study, 1 series and multiple sequences over time durig a day.
# Author: Sai Charan Ramireddy
# Date Created: November 11 2025
# Tools/Support Used: OpenAI Codex for code generation assistance, referred pandas, pydicom modules
# ======================================================================


In [ ]:
# ==========================================================
# LAYER 1 — FILE HANDLING (Nested ZIP Structure)
# ==========================================================

import os
import zipfile

# -------------------------------
# CONFIGURATION
# -------------------------------
MAIN_ZIP_PATH = "/content/drive/MyDrive/Files.zip"
BASE_EXTRACT_PATH = "/content/extracted_data"

# Create extraction directory
os.makedirs(BASE_EXTRACT_PATH, exist_ok=True)


# -------------------------------
# STEP 1 — Extract main ZIP
# -------------------------------
print(" Extracting main ZIP...")

with zipfile.ZipFile(MAIN_ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(BASE_EXTRACT_PATH)

print(" Main ZIP extracted.")


# -------------------------------
# STEP 2 — Find and extract nested ZIP files
# -------------------------------
print(" Searching for nested ZIP files...")

nested_zip_paths = []

for root, dirs, files in os.walk(BASE_EXTRACT_PATH):
    for file in files:
        if file.lower().endswith(".zip"):
            nested_zip_paths.append(os.path.join(root, file))

print(f" Found {len(nested_zip_paths)} nested ZIP file(s).")

for zip_path in nested_zip_paths:
    extract_to = os.path.splitext(zip_path)[0]  # remove .zip
    os.makedirs(extract_to, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

    print(f" Extracted: {zip_path}")

print(" All nested ZIP files extracted.")


# -------------------------------
# STEP 3 — Locate IMAGES folders
# -------------------------------
print(" Locating IMAGES folders...")

CT_IMAGES_PATH = None
MRT_IMAGES_PATH = None

for root, dirs, files in os.walk(BASE_EXTRACT_PATH):
    if os.path.basename(root).upper() == "IMAGES":
        full_path_upper = root.upper()

        if "CT" in full_path_upper:
            CT_IMAGES_PATH = root
        elif "MRT" in full_path_upper:
            MRT_IMAGES_PATH = root

print("\n Final Paths:")
print("CT Images Path :", CT_IMAGES_PATH)
print("MRT Images Path:", MRT_IMAGES_PATH)


# -------------------------------
# VALIDATION
# -------------------------------
if not CT_IMAGES_PATH:
    print(" CT IMAGES folder not found.")
if not MRT_IMAGES_PATH:
    print(" MRT IMAGES folder not found.")

if CT_IMAGES_PATH and MRT_IMAGES_PATH:
    print("\n LAYER 1 COMPLETE — File handling successful.")

In [ ]:
# -------------------------------
# STEP 3 — Locate IMAGES folders (Robust Version)
# -------------------------------
print(" Locating IMAGES folders...")

CT_IMAGES_PATH = None
MRT_IMAGES_PATH = None

for root, dirs, files in os.walk(BASE_EXTRACT_PATH):

    if os.path.basename(root).upper() == "IMAGES":

        parent_folder = os.path.basename(os.path.dirname(root)).upper()

        if "CT" in parent_folder:
            CT_IMAGES_PATH = root

        elif "MRT" in parent_folder:
            MRT_IMAGES_PATH = root


print("\n Final Paths:")
print("CT Images Path :", CT_IMAGES_PATH)
print("MRT Images Path:", MRT_IMAGES_PATH)


# -------------------------------
# VALIDATION
# -------------------------------
if not CT_IMAGES_PATH:
    print(" CT IMAGES folder not found.")

if not MRT_IMAGES_PATH:
    print(" MRT IMAGES folder not found.")

if CT_IMAGES_PATH and MRT_IMAGES_PATH:
    print("\n LAYER 1 COMPLETE — File handling successful.")

In [ ]:
pip install pydicom

In [ ]:
# ==========================================================
# LAYER 2 — PRINT DICOM HEADERS (CT & MRT)
# ==========================================================

import os
import pydicom


def get_sample_dicom(images_path):
    """Return first valid DICOM file found in folder."""
    for root, _, files in os.walk(images_path):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                ds = pydicom.dcmread(file_path, stop_before_pixels=True)
                return file_path, ds
            except Exception:
                continue
    return None, None


def print_dicom_header(title, images_path):
    """Print formatted DICOM header for dataset."""
    print("\n" + "="*70)
    print(f" DATASET: {title}")
    print("="*70)

    sample_path, ds = get_sample_dicom(images_path)

    if ds is None:
        print(" No valid DICOM files found.")
        return

    print(f" Sample File: {sample_path}")
    print("-"*70)

    for elem in ds:
        if elem.VR == "SQ":
            continue  # Skip sequences for cleaner output

        tag_name = elem.keyword if elem.keyword else str(elem.tag)
        print(f"{tag_name:40} : {elem.value}")


# -------------------------------
# PRINT HEADERS
# -------------------------------

print_dicom_header("IMAGINE CT", CT_IMAGES_PATH)
print_dicom_header("IMAGINE MRT", MRT_IMAGES_PATH)

In [ ]:
# ==========================================================
# LAYER 2 — PRINT DICOM HEADERS (CT & MRT)
# ==========================================================

import os
import pydicom


def get_sample_dicom(images_path):
    """Return first valid DICOM file found in folder."""
    for root, _, files in os.walk(images_path):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                ds = pydicom.dcmread(file_path, stop_before_pixels=True)
                return file_path, ds
            except Exception:
                continue
    return None, None


def print_dicom_header(title, images_path):
    """Print formatted DICOM header for dataset."""
    print("\n" + "="*70)
    print(f" DATASET: {title}")
    print("="*70)

    sample_path, ds = get_sample_dicom(images_path)

    if ds is None:
        print(" No valid DICOM files found.")
        return

    print(f"📄 Sample File: {sample_path}")
    print("-"*70)

    for elem in ds:
        if elem.VR == "SQ":
            continue  # Skip sequences for cleaner output

        tag_name = elem.keyword if elem.keyword else str(elem.tag)
        print(f"{tag_name:40} : {elem.value}")


# -------------------------------
# PRINT HEADERS
# -------------------------------

print_dicom_header("IMAGINE CT", CT_IMAGES_PATH)
print_dicom_header("IMAGINE MRT", MRT_IMAGES_PATH)

In [ ]:
import pandas as pd
from tqdm import tqdm


In [ ]:
# ==========================================================
# LAYER — TARGETED METADATA EXTRACTION (CT & MRT)
# Extract only user-selected headers for quality/AI criteria
# ==========================================================

import os
import pydicom
import pandas as pd
from tqdm import tqdm


def find_all_dicoms(folder):
    """Recursively find and verify all DICOM files in a folder."""
    dicom_files = []
    for root, _, files in os.walk(folder):
        for f in files:
            file_path = os.path.join(root, f)

            # Skip obvious non-DICOM formats
            if f.lower().endswith(('.txt', '.csv', '.json', '.xml', '.zip')):
                continue

            try:
                pydicom.dcmread(file_path, stop_before_pixels=True)
                dicom_files.append(file_path)
            except Exception:
                continue

    return dicom_files


def safe_get(ds, key):
    """
    Robust getter:
    - key can be DICOM keyword string (e.g. 'StudyDate')
    - or a tag tuple (e.g. (0x0040, 0x0254))
    Returns plain python value or None.
    """
    try:
        val = ds.get(key, None)
        # If pydicom DataElement returned (happens for tag tuple access sometimes)
        if hasattr(val, "value"):
            val = val.value
        # Convert multi-values to readable strings
        if isinstance(val, (list, tuple)):
            return "\\".join(map(str, val))
        return None if val is None else str(val)
    except Exception:
        return None


def extract_selected_headers(dicom_path, headers, extra_cols=None):
    """Extract only selected headers + optional extra cols like file path."""
    extra_cols = extra_cols or {}
    row = dict(extra_cols)

    try:
        ds = pydicom.dcmread(dicom_path, stop_before_pixels=True)
        for h in headers:
            row[h if isinstance(h, str) else str(h)] = safe_get(ds, h)
    except Exception as e:
        row["_read_error"] = str(e)

    return row


def build_dataset(images_path, selected_headers, dataset_name):
    """Create a DataFrame for a dataset using selected headers."""
    dicom_files = find_all_dicoms(images_path)
    print(f" {dataset_name}: Found {len(dicom_files)} DICOM files")

    rows = []
    for path in tqdm(dicom_files, desc=f"Extracting {dataset_name} headers"):
        rows.append(
            extract_selected_headers(
                path,
                selected_headers,
                extra_cols={
                    "Dataset": dataset_name,
                    "FilePath": path
                }
            )
        )

    df = pd.DataFrame(rows)

    # Helpful: bring key identifiers to front if present
    preferred_front = ["Dataset", "FilePath", "PatientID", "StudyInstanceUID", "SeriesInstanceUID", "SOPInstanceUID"]
    cols = [c for c in preferred_front if c in df.columns] + [c for c in df.columns if c not in preferred_front]
    df = df[cols]

    return df


# -------------------------------
# 1) SELECTED HEADER LISTS
# -------------------------------
CT_HEADERS =  [
  'BitsStored',
  'FrameOfReferenceUID',
  'ContrastBolusAgent',
  'ContrastBolusVolume',
  'ContrastBolusIngredient',
  'ContrastBolusIngredientConcentration',
  'ContrastBolusTotalDose',
  'PatientSex',
  'ImageOrientationPatient',
  'BitsStored',
  'FrameOfReferenceUID',
  'InstanceCreationDate',
  'PerformedProcedureStepDescription',
  'Modality',
  'PatientSize',
  'AcquisitionDate',
  'SliceThickness',
  'PatientID',
  'PhotometricInterpretation',
  'InstitutionName',
  'Rows',
  'Columns',
  'KVP',
  'BodyPartExamined',
  'LargestImagePixelValue',
  'CTDIvol',
  'XRayTubeCurrent',
  'SeriesDescription',
  'MagneticFieldStrength',
  'StudyDescription',
  'PatientWeight',
  'ProtocolName',
  'SeriesTime'
]

MRT_HEADERS =[
  'ScanningSequence',
  'BitsAllocated',
  'ContrastBolusAgent',
  'ContrastBolusVolume',
  'ScanOptions',
  'PatientSex',
  'ImageOrientationPatient',
  'BitsStored',
  'Modality',
  'PatientSize',
  'SliceThickness',
  'PatientID',
  'AcquisitionDate',
  'BodyPartExamined',
  'ImageType',
  'Rows',
  'LargestImagePixelValue',
  'Columns',
  'SeriesDescription',
  'MagneticFieldStrength',
  'StudyDescription',
  'PatientWeight',
  'ProtocolName'
]
# -------------------------------
# 2) BUILD DATASETS
# -------------------------------
df_ct  = build_dataset(CT_IMAGES_PATH,  CT_HEADERS,  "IMAGINE_CT")
df_mrt = build_dataset(MRT_IMAGES_PATH, MRT_HEADERS, "IMAGINE_MRT")

# -------------------------------
# 3) SAVE OUTPUTS
# -------------------------------
OUTPUT_DIR = "/content/drive/MyDrive/dicom_quality_exports"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ct_csv  = os.path.join(OUTPUT_DIR, "ct_selected_headers.csv")
mrt_csv = os.path.join(OUTPUT_DIR, "mrt_selected_headers.csv")

df_ct.to_csv(ct_csv, index=False)
df_mrt.to_csv(mrt_csv, index=False)

print("\n Saved:")
print("CT :", ct_csv)
print("MRT:", mrt_csv)

# Optional combined export (useful for unified AI pipeline later)
combined_csv = os.path.join(OUTPUT_DIR, "ct_mrt_selected_headers_combined.csv")
pd.concat([df_ct, df_mrt], ignore_index=True).to_csv(combined_csv, index=False)
print("COMBINED:", combined_csv)

# Optional download links (Colab)
from google.colab import files
files.download(ct_csv)
files.download(mrt_csv)
files.download(combined_csv)
